# Dagbani ASR Phase 2 — Budget-Aware Whisper Training

This notebook trains an owned Dagbani ASR model from `openai/whisper-small`.
It consumes the **full, release-ready Phase 1 manifest** and runs one stage at
a time: `smoke`, `supervised`, `domain_adaptation`, or `pseudo_label`.

The default is a 20-step smoke run. Full stages refuse to start when the Phase
1 readiness file is missing, when the test split leaks into training, or when
the audited manifest contains undecoded supervised audio.

The notebook supports one GPU and genuine notebook-launched DDP. It never uses
`device_map="auto"` as a substitute for data parallelism.

## 1. Install dependencies without replacing Kaggle's PyTorch/CUDA build

In [ ]:
import importlib.util
import subprocess
import sys

INSTALL_DEPS = True
REQUIRED = {
    "datasets": "datasets[audio]>=3.2,<5",
    "transformers": "transformers>=4.48,<6",
    "accelerate": "accelerate>=1.2,<3",
    "huggingface_hub": "huggingface_hub>=0.27,<2",
    "peft": "peft>=0.14,<1",
    "bitsandbytes": "bitsandbytes>=0.45,<1",
    "jiwer": "jiwer>=3.0,<5",
    "soundfile": "soundfile>=0.12,<1",
    "librosa": "librosa>=0.10,<1",
    "pyarrow": "pyarrow>=17,<25",
}
missing = [spec for module, spec in REQUIRED.items() if importlib.util.find_spec(module) is None]
if INSTALL_DEPS and missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Dependencies ready. Restart the kernel only if pip explicitly requests it.")

## 2. Stage configuration, secrets, and hard safety gates

In [ ]:
from __future__ import annotations

import dataclasses
import gzip
import hashlib
import inspect
import io
import json
import math
import os
import random
import re
import shutil
import string
import subprocess
import textwrap
import time
import unicodedata
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Iterable, Iterator

import numpy as np
import pandas as pd


def kaggle_secret(name: str) -> str | None:
    value = os.getenv(name)
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None


@dataclass
class TrainConfig:
    stage: str = "smoke"  # smoke | supervised | domain_adaptation | pseudo_label | medium_pilot | medium_full
    base_model_id: str = "openai/whisper-small"
    manifest_repo_id: str = os.getenv("DAGBANI_MANIFEST_REPO", "")
    model_repo_id: str = os.getenv("DAGBANI_MODEL_REPO", "")
    hf_token: str | None = field(default_factory=lambda: kaggle_secret("HF_TOKEN"))
    work_dir: str = "/kaggle/working/dagbani_asr_phase2"
    manifest_local_dir: str = os.getenv("DAGBANI_MANIFEST_DIR", "")
    seed: int = 42
    micro_batch_size: int = 4
    effective_batch_size: int = 32
    learning_rate: float = 1e-4
    warmup_steps: int = 250
    supervised_max_steps: int = 2_500
    domain_max_steps: int = 750
    pseudo_max_steps: int = 750
    smoke_steps: int = 20
    medium_pilot_steps: int = 100
    medium_full_max_steps: int = 2_500
    medium_gate_file: str = "/kaggle/working/dagbani_asr_phase2/evaluations/medium_gate.json"
    save_steps: int = 250
    eval_steps: int = 250
    major_checkpoint_steps: tuple[int, ...] = (1_000, 2_000, 2_500)
    eval_generation_limit: int | None = None
    max_session_minutes: int = 540
    stop_buffer_minutes: int = 20
    dataloader_workers: int = 2
    use_ddp: str = "auto"  # auto | one | two
    training_mode: str = "full"  # full | lora
    waxal_batch_share: float = 0.60
    pseudo_batch_share: float = 0.25
    pseudo_initial_hours: float = 20.0
    pseudo_calibration_rows: int = 128
    pseudo_max_new_tokens: int = 225
    resume: bool = True
    upload_full_state: bool = True
    run_training: bool = False
    run_full_generation_eval: bool = False
    run_external_generation_eval: bool = False
    external_eval_limit: int | None = None
    run_pseudo_labelling: bool = False
    allow_remote_audio_downloads: bool = False

    def validate(self) -> None:
        allowed = {"smoke", "supervised", "domain_adaptation", "pseudo_label", "medium_pilot", "medium_full"}
        if self.stage not in allowed:
            raise ValueError(f"stage must be one of {sorted(allowed)}")
        if self.training_mode not in {"full", "lora"}:
            raise ValueError("training_mode must be full or lora")
        if self.use_ddp not in {"auto", "one", "two"}:
            raise ValueError("use_ddp must be auto, one, or two")
        if self.stage in {"supervised", "domain_adaptation", "pseudo_label", "medium_pilot", "medium_full"} and not self.manifest_repo_id and not self.manifest_local_dir:
            raise ValueError("Full stages require DAGBANI_MANIFEST_REPO or DAGBANI_MANIFEST_DIR")
        if self.run_training and self.run_pseudo_labelling:
            raise ValueError("Generate pseudo labels and train in separate runs; enable only one action")
        if (self.run_training or self.run_pseudo_labelling) and not self.hf_token:
            raise ValueError("HF_TOKEN is required for datasets and private checkpoint persistence")
        if self.run_training and not self.model_repo_id:
            raise ValueError("Set DAGBANI_MODEL_REPO before training")
        if not (0.60 <= self.waxal_batch_share <= 1.0):
            raise ValueError("waxal_batch_share must preserve at least 60% WAXAL batches")
        if self.stage == "medium_full":
            gate_path = Path(self.medium_gate_file)
            if not gate_path.exists() or not json.loads(gate_path.read_text(encoding="utf-8")).get("allow_medium_full_run"):
                raise ValueError("medium_full is locked until a passing medium_gate.json is created")

    def max_steps(self) -> int:
        return {
            "smoke": self.smoke_steps,
            "supervised": self.supervised_max_steps,
            "domain_adaptation": self.domain_max_steps,
            "pseudo_label": self.pseudo_max_steps,
            "medium_pilot": self.medium_pilot_steps,
            "medium_full": self.medium_full_max_steps,
        }[self.stage]


cfg = TrainConfig()
cfg.validate()
WORK_DIR = Path(cfg.work_dir)
OUTPUT_DIR = WORK_DIR / f"{cfg.stage}_output"
CACHE_DIR = WORK_DIR / "hf_cache"
EVAL_DIR = WORK_DIR / "evaluations"
for directory in (WORK_DIR, OUTPUT_DIR, CACHE_DIR, EVAL_DIR):
    directory.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(CACHE_DIR)
os.environ["HF_DATASETS_CACHE"] = str(CACHE_DIR / "datasets")
os.environ["TRANSFORMERS_CACHE"] = str(CACHE_DIR / "models")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print(json.dumps(asdict(cfg) | {"hf_token": "<set>" if cfg.hf_token else "<missing>"}, indent=2))
print("Training is OFF by default. Complete all preflight cells before setting run_training=True.")

## 3. Load and validate the Phase 1 evidence contract

In [ ]:
CANONICAL_COLUMNS = [
    "sample_id", "source", "source_version", "audio_locator",
    "transcript_raw", "transcript_eval", "speaker_id", "duration_s",
    "domain", "split", "license", "audio_hash", "is_pseudo",
    "pseudo_confidence",
]
PUNCT_TRANSLATION = str.maketrans({char: " " for char in string.punctuation + "“”‘’…–—"})


def normalize_eval(text: Any) -> str:
    value = unicodedata.normalize("NFC", str(text or "")).lower().translate(PUNCT_TRANSLATION)
    return " ".join(value.split())


def artifact_path(filename: str) -> Path:
    if cfg.manifest_local_dir:
        candidate = Path(cfg.manifest_local_dir) / filename
        if candidate.exists():
            return candidate
        nested = Path(cfg.manifest_local_dir) / "phase1" / filename
        if nested.exists():
            return nested
    if not cfg.manifest_repo_id:
        raise FileNotFoundError(filename)
    from huggingface_hub import hf_hub_download
    return Path(hf_hub_download(
        cfg.manifest_repo_id, filename=f"phase1/{filename}", repo_type="dataset",
        token=cfg.hf_token, cache_dir=str(CACHE_DIR),
    ))


manifest_df = pd.DataFrame(columns=CANONICAL_COLUMNS)
readiness = {"audit_mode": "missing", "release_ready": False, "warnings": ["No Phase 1 manifest loaded"]}
if cfg.manifest_repo_id or cfg.manifest_local_dir:
    manifest_df = pd.read_parquet(artifact_path("accepted_manifest.parquet"))
    readiness = json.loads(artifact_path("readiness.json").read_text(encoding="utf-8"))
    missing_columns = set(CANONICAL_COLUMNS) - set(manifest_df.columns)
    if missing_columns:
        raise ValueError(f"Manifest contract is missing: {sorted(missing_columns)}")


def assert_manifest_safe(table: pd.DataFrame, require_release_ready: bool) -> None:
    if require_release_ready and not readiness.get("release_ready"):
        raise RuntimeError(f"Phase 1 is not release-ready: {readiness}")
    supervised = table[table["split"].isin(["train", "validation", "test", "external_test"])]
    if require_release_ready and supervised.empty:
        raise RuntimeError("No supervised rows were admitted")
    if not supervised.empty:
        if supervised["transcript_raw"].fillna("").eq("").any():
            raise RuntimeError("A supervised manifest row has no transcript")
        if require_release_ready and pd.to_numeric(supervised["duration_s"], errors="coerce").isna().any():
            raise RuntimeError("A supervised manifest row was not audio-audited")
        audio_rows = supervised[supervised["audio_hash"].fillna("").astype(str).ne("")]
        if (audio_rows.groupby("audio_hash")["split"].nunique() > 1).any():
            raise RuntimeError("Cross-split audio_hash leakage remains")

        speaker_rows = supervised[
            supervised["speaker_id"].fillna("").astype(str).ne("")
            & supervised["speaker_id"].ne("unknown")
        ]
        for speaker_id, group in speaker_rows.groupby("speaker_id"):
            if group["split"].nunique() <= 1:
                continue
            official_waxal_overlap = bool(
                group["source"].eq("waxal_dag_asr").all()
                and set(group["split"].unique()).issubset({"train", "validation", "test"})
                and ("split_policy" not in group or group["split_policy"].eq("official").all())
            )
            if not official_waxal_overlap:
                raise RuntimeError(f"Cross-split speaker_id leakage remains for {speaker_id}")
    held_out = table[table["split"].isin(["test", "external_test"])]
    if not held_out.empty and held_out["is_pseudo"].fillna(False).any():
        raise RuntimeError("Held-out rows may never be pseudo-labelled")


assert_manifest_safe(manifest_df, require_release_ready=cfg.stage != "smoke")
print(f"Manifest rows: {len(manifest_df):,}; readiness={readiness}")

## 4. Dataset loaders preserve official splits and audited IDs

In [ ]:
import requests
from datasets import Audio, Dataset, DatasetDict, concatenate_datasets, interleave_datasets, load_dataset

HF_SOURCE_REGISTRY = {
    "waxal_dag_asr": ("google/WaxalNLP", "dag_asr"),
    "dagbani_bible": ("ghananlpcommunity/dagbani-bible-audio-text-tts", None),
    "ghana_speech_dag": ("ghananlpcommunity/ghana-speech", "Dagbani_dag"),
    "navigation_dagbani": ("ghananlpcommunity/navigation-corpus-dagbani-speech", None),
    "health_unicef_dagbani": ("ghananlpcommunity/ghana-nlp-health-UNICEF-asr-dagbani", None),
    "youth_conversations_dagbani": ("ghananlpcommunity/youth-conversations-dag", None),
}


def pick_column(columns: Iterable[str], candidates: Iterable[str]) -> str | None:
    available = set(columns)
    return next((name for name in candidates if name in available), None)


def standardize_dataset(
    dataset: Dataset,
    source: str,
    text_override: dict[str, str] | None = None,
    accepted_ids: set[str] | None = None,
) -> Dataset:
    audio_col = pick_column(dataset.column_names, ("audio", "speech"))
    text_col = pick_column(dataset.column_names, ("transcription", "sentence", "text", "transcript"))
    id_col = pick_column(dataset.column_names, ("id", "sample_id", "path", "file", "filename"))
    if not audio_col or not id_col:
        raise ValueError(f"{source} does not expose auditable audio and id columns: {dataset.column_names}")
    if text_override is None and not text_col:
        raise ValueError(f"{source} has no transcription column")
    if audio_col != "audio":
        dataset = dataset.rename_column(audio_col, "audio")
    if id_col != "sample_id":
        dataset = dataset.rename_column(id_col, "sample_id")
    if accepted_ids is not None:
        if not accepted_ids:
            return dataset.select([])
        dataset = dataset.filter(
            lambda sample_id: str(sample_id) in accepted_ids,
            input_columns=["sample_id"],
            desc="Applying audited manifest IDs without decoding audio",
        )
    if text_override is not None:
        dataset = dataset.map(
            lambda sample_id: {"sentence": text_override[str(sample_id)]},
            input_columns=["sample_id"],
            desc="Attaching audited pseudo transcripts",
        )
    elif text_col != "sentence":
        dataset = dataset.rename_column(text_col, "sentence")
    dataset = dataset.map(
        lambda sample_id: {"sample_id": str(sample_id), "source": source},
        input_columns=["sample_id"],
        desc="Standardizing sample IDs",
    )
    return dataset.cast_column("audio", Audio(sampling_rate=16_000))


def allowed_ids(source: str, split: str) -> set[str]:
    if manifest_df.empty:
        return set()
    rows = manifest_df[(manifest_df["source"] == source) & (manifest_df["split"] == split)]
    return set(rows["sample_id"].astype(str))


def filter_to_ids(dataset: Dataset, ids: set[str]) -> Dataset:
    if not ids:
        return dataset.select([])
    return dataset.filter(
        lambda sample_id: str(sample_id) in ids,
        input_columns=["sample_id"],
        desc="Applying audited manifest IDs without decoding audio",
    )


def load_waxal(split: str) -> Dataset:
    dataset = load_dataset("google/WaxalNLP", "dag_asr", split=split, token=cfg.hf_token, cache_dir=str(CACHE_DIR))
    ids = allowed_ids("waxal_dag_asr", split)
    if not ids and cfg.stage != "smoke":
        raise RuntimeError(f"No audited WAXAL IDs for {split}")
    return standardize_dataset(dataset, "waxal_dag_asr", accepted_ids=ids or None)


def load_waxal_smoke(split: str, rows: int) -> Dataset:
    '''Materialize only a tiny streamed shard so smoke mode does not download 90h.'''
    stream = load_dataset("google/WaxalNLP", "dag_asr", split=split, streaming=True, token=cfg.hf_token)
    materialized = []
    for index, record in enumerate(stream):
        if index >= rows:
            break
        audio = record["audio"]
        if hasattr(audio, "get_all_samples"):
            samples = audio.get_all_samples()
            array = samples.data.detach().cpu().numpy().squeeze()
            sample_rate = int(samples.sample_rate)
        else:
            array = np.asarray(audio["array"], dtype=np.float32)
            sample_rate = int(audio["sampling_rate"])
        materialized.append({
            "sample_id": str(record.get("id", f"{split}-{index}")),
            "source": "waxal_dag_asr", "sentence": str(record.get("transcription", "")),
            "audio": {"array": array, "sampling_rate": sample_rate},
        })
    if not materialized:
        raise RuntimeError(f"Unable to stream WAXAL {split} smoke rows")
    return Dataset.from_list(materialized).cast_column("audio", Audio(sampling_rate=16_000))


def load_domain_source(source: str, split: str) -> Dataset | None:
    repo_id, config_name = HF_SOURCE_REGISTRY[source]
    rows = manifest_df[(manifest_df["source"] == source) & (manifest_df["split"] == split)].copy()
    if rows.empty:
        return None
    try:
        parts = []
        origin_values = rows["origin_split"].fillna("train").astype(str) if "origin_split" in rows else pd.Series(["train"] * len(rows))
        rows = rows.assign(_origin_split=origin_values.values)
        for origin_split, origin_rows in rows.groupby("_origin_split"):
            dataset = load_dataset(
                repo_id, config_name, split=origin_split,
                token=cfg.hf_token, cache_dir=str(CACHE_DIR),
            )
            ids = set(origin_rows["sample_id"].astype(str))
            part = standardize_dataset(dataset, source, accepted_ids=ids)
            if len(part):
                parts.append(part)
        return concatenate_datasets(parts) if parts else None
    except Exception as exc:
        print(f"Skipping {source}: {type(exc).__name__}: {exc}")
        return None


def load_manifest_locator_source(source: str, split: str) -> Dataset | None:
    rows = manifest_df[(manifest_df["source"] == source) & (manifest_df["split"] == split)].copy()
    if rows.empty:
        return None
    cache = WORK_DIR / "domain_audio" / source
    cache.mkdir(parents=True, exist_ok=True)
    usable = []
    for row in rows.to_dict("records"):
        locator = str(row["audio_locator"])
        resolved = Path(locator)
        if re.match(r"https?://", locator):
            if not cfg.allow_remote_audio_downloads:
                continue
            suffix = Path(locator.split("?", 1)[0]).suffix or ".audio"
            resolved = cache / f"{hashlib.sha256(locator.encode()).hexdigest()}{suffix}"
            if not resolved.exists():
                response = requests.get(locator, timeout=120)
                response.raise_for_status()
                resolved.write_bytes(response.content)
        if locator.startswith("hf://") or not resolved.exists():
            continue
        usable.append({
            "sample_id": str(row["sample_id"]), "source": source,
            "sentence": str(row["transcript_raw"]), "audio": str(resolved),
        })
    if not usable:
        print(f"Skipping {source}: audited rows exist but no local/allowed audio locator is resolvable")
        return None
    return Dataset.from_list(usable).cast_column("audio", Audio(sampling_rate=16_000))


def make_training_data(config: TrainConfig) -> tuple[Dataset, Dataset]:
    if config.stage == "smoke":
        return load_waxal_smoke("train", 64), load_waxal_smoke("validation", 32)
    train = load_waxal("train")
    validation = load_waxal("validation")
    if config.stage in {"supervised", "medium_pilot", "medium_full"}:
        return train, validation
    if config.stage == "domain_adaptation":
        extras = []
        audited_sources = sorted(set(manifest_df.loc[manifest_df["split"] == "train", "source"].astype(str)))
        for source in audited_sources:
            if source == "waxal_dag_asr":
                continue
            candidate = (
                load_domain_source(source, "train")
                if source in HF_SOURCE_REGISTRY
                else load_manifest_locator_source(source, "train")
            )
            if candidate is not None and len(candidate):
                extras.append(candidate)
        if not extras:
            raise RuntimeError("No audited domain-adaptation source is loadable")
        extra = concatenate_datasets(extras).shuffle(seed=config.seed)
        mixed = interleave_datasets(
            [train.shuffle(seed=config.seed), extra],
            probabilities=[config.waxal_batch_share, 1.0 - config.waxal_batch_share],
            seed=config.seed, stopping_strategy="all_exhausted",
        )
        return mixed, validation
    if config.stage == "pseudo_label":
        pseudo_path = artifact_path("pseudo_manifest.parquet")
        pseudo = pd.read_parquet(pseudo_path)
        if pseudo.empty or (pseudo["split"] == "test").any():
            raise RuntimeError("Pseudo manifest is empty or contains test rows")
        pseudo_text = dict(zip(pseudo["sample_id"].astype(str), pseudo["transcript_raw"].astype(str)))
        raw = load_dataset("google/WaxalNLP", "dag_asr", split="unlabeled", token=config.hf_token, cache_dir=str(CACHE_DIR))
        raw = standardize_dataset(
            raw, "waxal_dag_asr_pseudo", text_override=pseudo_text,
            accepted_ids=set(pseudo_text),
        )
        mixed = interleave_datasets(
            [train.shuffle(seed=config.seed), raw.shuffle(seed=config.seed)],
            probabilities=[1.0 - config.pseudo_batch_share, config.pseudo_batch_share],
            seed=config.seed, stopping_strategy="all_exhausted",
        )
        return mixed, validation
    raise AssertionError(config.stage)


print("Dataset loaders defined. They do not access the test split during training.")

## 5. Whisper processor and exact Dagbani glyph round-trip

In [ ]:
from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained(cfg.base_model_id, task="transcribe", cache_dir=str(CACHE_DIR), token=cfg.hf_token)
glyph_probe = "n nyɛla bɛ paɣaŋa mini ɔ ka ʒɛm shɛli"
token_ids = processor.tokenizer(glyph_probe).input_ids
decoded_probe = processor.tokenizer.decode(token_ids, skip_special_tokens=True).strip()
if unicodedata.normalize("NFC", decoded_probe) != unicodedata.normalize("NFC", glyph_probe):
    raise RuntimeError(f"Whisper tokenizer damaged Dagbani glyphs: {decoded_probe!r}")
if getattr(processor.tokenizer, "unk_token_id", None) in token_ids:
    raise RuntimeError("Unexpected <unk> token in Dagbani glyph probe")
print("Whisper byte-level tokenizer round-trip passed:", decoded_probe)

## 6. Dynamic feature collation — no full-corpus mel cache

In [ ]:
import torch
from dataclasses import dataclass


@dataclass
class SpeechSeq2SeqCollator:
    processor: Any

    def __call__(self, features: list[dict[str, Any]]) -> dict[str, torch.Tensor]:
        input_features = []
        label_features = []
        for feature in features:
            audio = feature["audio"]
            if hasattr(audio, "get_all_samples"):
                samples = audio.get_all_samples()
                array = samples.data.detach().cpu().numpy().squeeze()
                sample_rate = int(samples.sample_rate)
            else:
                array = np.asarray(audio["array"], dtype=np.float32)
                sample_rate = int(audio["sampling_rate"])
            inputs = self.processor.feature_extractor(array, sampling_rate=sample_rate)
            input_features.append({"input_features": inputs.input_features[0]})
            label_features.append({"input_ids": self.processor.tokenizer(str(feature["sentence"])).input_ids})
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        labels = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        label_ids = labels["input_ids"].masked_fill(labels["attention_mask"].ne(1), -100)
        decoder_start = self.processor.tokenizer.bos_token_id
        if decoder_start is not None and (label_ids[:, 0] == decoder_start).all().item():
            label_ids = label_ids[:, 1:]
        batch["labels"] = label_ids
        return batch


collator = SpeechSeq2SeqCollator(processor)
print("Dynamic collator ready; features are computed only for the current batch.")

## 7. Telemetry, wall-clock stopping, and private resume checkpoints

In [ ]:
from transformers import TrainerCallback
from huggingface_hub import HfApi


class BudgetTelemetryCallback(TrainerCallback):
    def __init__(self, config: TrainConfig, output_dir: Path):
        self.config = config
        self.output_dir = output_dir
        self.started = None
        self.log_path = output_dir / "telemetry.jsonl"

    def on_train_begin(self, args, state, control, **kwargs):
        self.started = time.monotonic()

    def on_step_end(self, args, state, control, **kwargs):
        if not state.is_world_process_zero or not self.started or state.global_step < 1:
            return control
        elapsed_min = (time.monotonic() - self.started) / 60
        step_min = elapsed_min / state.global_step
        eta_min = step_min * max(0, state.max_steps - state.global_step)
        payload = {
            "step": state.global_step, "max_steps": state.max_steps,
            "elapsed_min": elapsed_min, "minutes_per_step": step_min,
            "eta_min": eta_min, "projected_total_min": elapsed_min + eta_min,
        }
        try:
            query = subprocess.check_output(
                ["nvidia-smi", "--query-gpu=index,utilization.gpu,memory.used,memory.total", "--format=csv,noheader,nounits"],
                text=True, timeout=10,
            ).strip()
            payload["gpu"] = query
        except Exception:
            payload["gpu"] = "unavailable"
        with self.log_path.open("a", encoding="utf-8") as handle:
            handle.write(json.dumps(payload) + "\n")
        if state.global_step == 20:
            print("20-step projection:", json.dumps(payload, indent=2))
            if payload["projected_total_min"] > self.config.max_session_minutes:
                print("Projected run exceeds the session budget; saving and stopping safely.")
                control.should_save = True
                control.should_training_stop = True
        if elapsed_min >= self.config.max_session_minutes - self.config.stop_buffer_minutes:
            print("Wall-clock safety buffer reached; saving and stopping safely.")
            control.should_save = True
            control.should_training_stop = True
        return control


class PrivateHubCheckpointCallback(TrainerCallback):
    '''Keep remote latest, indexed best two, and named major checkpoints.'''
    def __init__(self, config: TrainConfig, output_dir: Path):
        self.config = config
        self.output_dir = output_dir
        self.api = HfApi(token=config.hf_token) if config.hf_token and config.model_repo_id else None
        self.eval_losses: dict[int, float] = {}
        self.futures = []

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics and "eval_loss" in metrics:
            self.eval_losses[int(state.global_step)] = float(metrics["eval_loss"])

    def _upload(self, folder: Path, remote_path: str, message: str) -> None:
        if not self.api or not folder.exists():
            return
        future = self.api.upload_folder(
            repo_id=self.config.model_repo_id, repo_type="model", folder_path=str(folder),
            path_in_repo=remote_path, commit_message=message, run_as_future=True,
        )
        self.futures.append(future)

    def on_save(self, args, state, control, **kwargs):
        if not state.is_world_process_zero:
            return control
        step = int(state.global_step)
        folder = Path(args.output_dir) / f"checkpoint-{step}"
        if self.config.upload_full_state:
            self._upload(folder, f"resume/{self.config.stage}/latest", f"Update {self.config.stage} resumable checkpoint at step {step}")
        ranked = sorted(self.eval_losses.items(), key=lambda item: item[1])[:2]
        if step in {item[0] for item in ranked}:
            self._upload(folder, f"resume/{self.config.stage}/best-step-{step}", f"Preserve {self.config.stage} top-two checkpoint step {step}")
        if step in self.config.major_checkpoint_steps:
            self._upload(folder, f"major/{self.config.stage}/checkpoint-{step}", f"Preserve {self.config.stage} major checkpoint step {step}")
        index = {
            "latest_step": step,
            "best_two": [{"step": key, "eval_loss": value} for key, value in ranked],
            "major_steps": list(self.config.major_checkpoint_steps),
        }
        index_path = self.output_dir / "checkpoint_index.json"
        index_path.write_text(json.dumps(index, indent=2), encoding="utf-8")
        if self.api:
            future = self.api.upload_file(
                repo_id=self.config.model_repo_id, repo_type="model", path_or_fileobj=str(index_path),
                path_in_repo=f"resume/{self.config.stage}/checkpoint_index.json", commit_message=f"{self.config.stage} checkpoint index step {step}",
                run_as_future=True,
            )
            self.futures.append(future)
        return control

    def wait(self) -> None:
        for future in self.futures:
            future.result()


print("Budget telemetry and resumable private checkpoint callbacks ready.")

## 8. Trainer factory: full fine-tuning by default, LoRA only as fallback

In [ ]:
from transformers import AutoModelForSpeechSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments, set_seed


def gpu_count_without_cuda_init() -> int:
    try:
        output = subprocess.check_output(["nvidia-smi", "-L"], text=True, timeout=10)
        return len([line for line in output.splitlines() if line.strip()])
    except Exception:
        return 0


def requested_world_size(config: TrainConfig) -> int:
    available = gpu_count_without_cuda_init()
    if config.use_ddp == "one":
        return 1
    if config.use_ddp == "two":
        if available < 2:
            raise RuntimeError("Two-GPU DDP requested but fewer than two GPUs are visible")
        return 2
    return 2 if available >= 2 else 1


def model_starting_point(config: TrainConfig) -> str:
    if config.stage in {"smoke", "supervised", "medium_pilot", "medium_full"}:
        return "openai/whisper-medium" if config.stage in {"medium_pilot", "medium_full"} else config.base_model_id
    return config.model_repo_id


def training_args(config: TrainConfig, world_size: int) -> Seq2SeqTrainingArguments:
    accumulation = max(1, math.ceil(config.effective_batch_size / (config.micro_batch_size * world_size)))
    kwargs = dict(
        output_dir=str(OUTPUT_DIR), per_device_train_batch_size=config.micro_batch_size,
        per_device_eval_batch_size=max(1, config.micro_batch_size),
        gradient_accumulation_steps=accumulation, learning_rate=config.learning_rate,
        warmup_steps=min(config.warmup_steps, max(1, config.max_steps() // 10)),
        max_steps=config.max_steps(), lr_scheduler_type="linear", fp16=True,
        gradient_checkpointing=True, remove_unused_columns=False,
        dataloader_num_workers=config.dataloader_workers, dataloader_pin_memory=True,
        save_strategy="steps", save_steps=min(config.save_steps, config.max_steps()),
        eval_steps=min(config.eval_steps, config.max_steps()), logging_steps=10,
        save_total_limit=3, load_best_model_at_end=True, metric_for_best_model="eval_loss",
        greater_is_better=False, predict_with_generate=False, report_to="none",
        seed=config.seed, data_seed=config.seed, ddp_find_unused_parameters=False,
        optim="adamw_bnb_8bit" if importlib.util.find_spec("bitsandbytes") else "adamw_torch_fused",
        push_to_hub=bool(config.model_repo_id), hub_model_id=config.model_repo_id or None,
        hub_private_repo=True, hub_token=config.hf_token, hub_strategy="end",
    )
    signature = inspect.signature(Seq2SeqTrainingArguments.__init__).parameters
    kwargs["eval_strategy" if "eval_strategy" in signature else "evaluation_strategy"] = "steps"
    return Seq2SeqTrainingArguments(**kwargs)


def build_model(config: TrainConfig):
    start = model_starting_point(config)
    model = AutoModelForSpeechSeq2Seq.from_pretrained(
        start, torch_dtype=torch.float16, low_cpu_mem_usage=True,
        cache_dir=str(CACHE_DIR), token=config.hf_token,
    )
    model.config.forced_decoder_ids = None
    model.generation_config.forced_decoder_ids = None
    model.generation_config.task = "transcribe"
    model.config.use_cache = False
    model.gradient_checkpointing_enable()
    if config.training_mode == "lora":
        from peft import LoraConfig, TaskType, get_peft_model
        lora = LoraConfig(
            r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
            target_modules=["q_proj", "v_proj", "out_proj"], task_type=TaskType.SEQ_2_SEQ_LM,
        )
        model = get_peft_model(model, lora)
        model.print_trainable_parameters()
    return model


def latest_local_checkpoint(output_dir: Path) -> str | None:
    checkpoints = sorted(output_dir.glob("checkpoint-*"), key=lambda path: int(path.name.split("-")[-1]))
    return str(checkpoints[-1]) if checkpoints else None


def resumable_checkpoint(config: TrainConfig, output_dir: Path) -> str | None:
    local = latest_local_checkpoint(output_dir)
    if local or not config.resume or not config.model_repo_id:
        return local
    from huggingface_hub import snapshot_download
    remote_root = WORK_DIR / f"remote_resume_{config.stage}"
    try:
        snapshot_download(
            repo_id=config.model_repo_id, repo_type="model", token=config.hf_token,
            allow_patterns=[f"resume/{config.stage}/latest/**"], local_dir=str(remote_root),
        )
        candidate = remote_root / "resume" / config.stage / "latest"
        if (candidate / "trainer_state.json").exists():
            print(f"Resuming from private Hub state: {candidate}")
            return str(candidate)
    except Exception as exc:
        print(f"No compatible remote resume state: {type(exc).__name__}: {exc}")
    return None


def train_process(config_dict: dict[str, Any]) -> dict[str, Any]:
    config = TrainConfig(**config_dict)
    set_seed(config.seed)
    world_size = int(os.environ.get("WORLD_SIZE", "1"))
    train_data, validation_data = make_training_data(config)
    model = build_model(config)
    telemetry = BudgetTelemetryCallback(config, OUTPUT_DIR)
    hub_checkpoints = PrivateHubCheckpointCallback(config, OUTPUT_DIR)
    args = training_args(config, world_size)
    trainer = Seq2SeqTrainer(
        model=model, args=args, train_dataset=train_data, eval_dataset=validation_data,
        data_collator=SpeechSeq2SeqCollator(processor), processing_class=processor,
        callbacks=[telemetry, hub_checkpoints],
    )
    resume_checkpoint = resumable_checkpoint(config, OUTPUT_DIR)
    result = trainer.train(resume_from_checkpoint=resume_checkpoint)
    trainer.save_model(str(OUTPUT_DIR / "final_model"))
    processor.save_pretrained(str(OUTPUT_DIR / "final_model"))
    if trainer.is_world_process_zero() and config.model_repo_id:
        trainer.push_to_hub(commit_message=f"Dagbani ASR {config.stage} completed at step {trainer.state.global_step}")
        hub_checkpoints.wait()
    return {"metrics": result.metrics, "global_step": trainer.state.global_step, "world_size": world_size}


world_size = requested_world_size(cfg)
effective_accumulation = math.ceil(cfg.effective_batch_size / (cfg.micro_batch_size * max(1, world_size)))
print({
    "visible_gpus": gpu_count_without_cuda_init(), "requested_world_size": world_size,
    "micro_batch": cfg.micro_batch_size, "gradient_accumulation": effective_accumulation,
    "effective_batch": cfg.micro_batch_size * world_size * effective_accumulation,
    "max_steps": cfg.max_steps(), "starting_model": model_starting_point(cfg),
})

## 9. Launch exactly one training stage

For two GPUs, this uses `accelerate.notebook_launcher`, which starts one process
per GPU. Do not run unrelated CUDA cells before this launcher. Set
`cfg.run_training=True` only after the preflight report is correct.

In [ ]:
training_result = None
if cfg.run_training:
    from accelerate import notebook_launcher
    from huggingface_hub import HfApi
    api = HfApi(token=cfg.hf_token)
    api.create_repo(cfg.model_repo_id, repo_type="model", private=True, exist_ok=True)
    if world_size > 1:
        notebook_launcher(train_process, args=(asdict(cfg),), num_processes=world_size, mixed_precision="fp16")
        training_result = {"status": "DDP child processes completed", "world_size": world_size}
    else:
        training_result = train_process(asdict(cfg))
    print(json.dumps(training_result, indent=2, default=str))
else:
    print("Dry preflight complete. Training was not launched.")

## 10. Full generation evaluation for final and preserved major checkpoints

In [ ]:
import jiwer


def strict_text(text: Any) -> str:
    return unicodedata.normalize("NFC", str(text or "")).strip()


def asr_metrics(references: list[str], hypotheses: list[str]) -> dict[str, float]:
    if len(references) != len(hypotheses):
        raise ValueError("Reference and hypothesis counts differ")
    return {
        "strict_wer": float(jiwer.wer([strict_text(x) for x in references], [strict_text(x) for x in hypotheses])),
        "strict_cer": float(jiwer.cer([strict_text(x) for x in references], [strict_text(x) for x in hypotheses])),
        "normalized_wer": float(jiwer.wer([normalize_eval(x) for x in references], [normalize_eval(x) for x in hypotheses])),
        "normalized_cer": float(jiwer.cer([normalize_eval(x) for x in references], [normalize_eval(x) for x in hypotheses])),
        "utterances": len(references),
    }


def evaluation_dataset(source: str, split: str) -> Dataset:
    if source == "waxal_dag_asr":
        return load_waxal(split)
    candidate = (
        load_domain_source(source, split)
        if source in HF_SOURCE_REGISTRY
        else load_manifest_locator_source(source, split)
    )
    if candidate is None or not len(candidate):
        raise RuntimeError(f"No resolvable audited rows for {source}/{split}")
    return candidate


def evaluate_generation(
    model_id_or_path: str,
    split: str = "validation",
    limit: int | None = None,
    source: str = "waxal_dag_asr",
) -> tuple[dict, pd.DataFrame]:
    from transformers import AutoModelForSpeechSeq2Seq
    model = AutoModelForSpeechSeq2Seq.from_pretrained(
        model_id_or_path, torch_dtype=torch.float16, low_cpu_mem_usage=True,
        token=cfg.hf_token, cache_dir=str(CACHE_DIR),
    ).to("cuda")
    model.eval()
    model.generation_config.forced_decoder_ids = None
    model.generation_config.task = "transcribe"
    dataset = evaluation_dataset(source, split)
    if limit is not None:
        dataset = dataset.select(range(min(limit, len(dataset))))
    references, hypotheses, sample_ids = [], [], []
    started = time.time()
    for index, row in enumerate(dataset, 1):
        audio = row["audio"]
        if hasattr(audio, "get_all_samples"):
            samples = audio.get_all_samples()
            array = samples.data.detach().cpu().numpy().squeeze()
            sample_rate = int(samples.sample_rate)
        else:
            array, sample_rate = audio["array"], audio["sampling_rate"]
        inputs = processor(array, sampling_rate=sample_rate, return_tensors="pt")
        features = inputs.input_features.to(device=model.device, dtype=model.dtype)
        with torch.inference_mode():
            tokens = model.generate(features, task="transcribe", max_new_tokens=225)
        hypothesis = processor.batch_decode(tokens, skip_special_tokens=True)[0]
        sample_ids.append(str(row["sample_id"]))
        references.append(strict_text(row["sentence"]))
        hypotheses.append(strict_text(hypothesis))
        if index % 50 == 0:
            print(f"Generated {index}/{len(dataset)} in {(time.time() - started) / 60:.1f} min")
    metrics = asr_metrics(references, hypotheses) | {
        "model": model_id_or_path, "source": source, "split": split,
        "elapsed_s": time.time() - started,
    }
    predictions = pd.DataFrame({"sample_id": sample_ids, "reference": references, "hypothesis": hypotheses})
    del model
    torch.cuda.empty_cache()
    return metrics, predictions


def resolve_major_checkpoint(step: int) -> str | None:
    local = OUTPUT_DIR / f"checkpoint-{step}"
    if local.exists():
        return str(local)
    if not cfg.model_repo_id:
        return None
    from huggingface_hub import snapshot_download
    download_root = EVAL_DIR / f"remote_major_{step}"
    try:
        snapshot_download(
            repo_id=cfg.model_repo_id, repo_type="model", token=cfg.hf_token,
            allow_patterns=[f"major/{cfg.stage}/checkpoint-{step}/**"], local_dir=str(download_root),
        )
        candidate = download_root / "major" / cfg.stage / f"checkpoint-{step}"
        return str(candidate) if (candidate / "config.json").exists() else None
    except Exception as exc:
        print(f"Major checkpoint {step} unavailable: {type(exc).__name__}: {exc}")
        return None


generation_metrics = []
if cfg.run_full_generation_eval:
    final_local = OUTPUT_DIR / "final_model"
    targets = [str(final_local) if final_local.exists() else cfg.model_repo_id]
    targets.extend(target for step in cfg.major_checkpoint_steps if (target := resolve_major_checkpoint(step)))
    for target in targets:
        metrics, predictions = evaluate_generation(target, limit=cfg.eval_generation_limit)
        generation_metrics.append(metrics)
        stem = Path(target).name
        predictions.to_parquet(EVAL_DIR / f"{stem}_validation_predictions.parquet", index=False)
        print(json.dumps(metrics, indent=2))
    (EVAL_DIR / "generation_metrics.json").write_text(json.dumps(generation_metrics, indent=2), encoding="utf-8")
else:
    print("Generation evaluation disabled. Enable it for selected/major checkpoints, not every 250-step save.")

external_metrics = []
if cfg.run_external_generation_eval:
    final_target = str(OUTPUT_DIR / "final_model") if (OUTPUT_DIR / "final_model").exists() else cfg.model_repo_id
    external_sources = sorted(set(manifest_df[manifest_df["source"] != "waxal_dag_asr"]["source"].astype(str)))
    for source in external_sources:
        source_rows = manifest_df[manifest_df["source"] == source]
        split = next((name for name in ("external_test", "test", "validation") if (source_rows["split"] == name).any()), None)
        if not split:
            continue
        try:
            metrics, predictions = evaluate_generation(
                final_target, split=split, limit=cfg.external_eval_limit, source=source,
            )
            external_metrics.append(metrics)
            predictions.to_parquet(EVAL_DIR / f"external_{source}_{split}.parquet", index=False)
            print(json.dumps(metrics, indent=2))
        except Exception as exc:
            external_metrics.append({"source": source, "split": split, "status": "unavailable", "error": f"{type(exc).__name__}: {exc}"})
    (EVAL_DIR / "external_metrics.json").write_text(json.dumps(external_metrics, indent=2), encoding="utf-8")
else:
    print("External-domain evaluation disabled. Enable it before a release decision.")

## 11. Confidence-calibrated pseudo-labelling of at most 20 initial hours

In [ ]:
def compression_ratio(text: str) -> float:
    raw = text.encode("utf-8")
    return len(raw) / max(1, len(gzip.compress(raw)))


def repetition_ratio(text: str, n: int = 3) -> float:
    tokens = normalize_eval(text).split()
    if len(tokens) < n:
        return 0.0
    grams = [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]
    return 1.0 - len(set(grams)) / max(1, len(grams))


def utterance_wer(reference: str, hypothesis: str) -> float:
    return float(jiwer.wer(normalize_eval(reference), normalize_eval(hypothesis)))


def decode_with_confidence(model, array: np.ndarray, sample_rate: int, num_beams: int) -> dict[str, Any]:
    inputs = processor(array, sampling_rate=sample_rate, return_tensors="pt")
    features = inputs.input_features.to(device=model.device, dtype=model.dtype)
    with torch.inference_mode():
        output = model.generate(
            features, task="transcribe", max_new_tokens=cfg.pseudo_max_new_tokens,
            num_beams=num_beams, return_dict_in_generate=True, output_scores=True,
        )
    beam_indices = getattr(output, "beam_indices", None)
    transition = model.compute_transition_scores(output.sequences, output.scores, beam_indices, normalize_logits=True)
    valid_scores = transition[transition < 0]
    average_logprob = float(valid_scores.mean().item()) if valid_scores.numel() else -99.0
    no_speech_id = processor.tokenizer.convert_tokens_to_ids("<|nospeech|>")
    first_logits = output.scores[0][0]
    no_speech_probability = float(torch.softmax(first_logits.float(), dim=-1)[no_speech_id].item())
    text = strict_text(processor.batch_decode(output.sequences, skip_special_tokens=True)[0])
    return {"text": text, "average_logprob": average_logprob, "no_speech_probability": no_speech_probability}


def calibrate_pseudo_thresholds(model, validation: Dataset, rows: int) -> dict[str, float]:
    scores = []
    for row in validation.select(range(min(rows, len(validation)))):
        audio = row["audio"]
        if hasattr(audio, "get_all_samples"):
            samples = audio.get_all_samples()
            array = samples.data.detach().cpu().numpy().squeeze()
            sample_rate = int(samples.sample_rate)
        else:
            array, sample_rate = np.asarray(audio["array"]), int(audio["sampling_rate"])
        greedy = decode_with_confidence(model, array, sample_rate, 1)
        beam = decode_with_confidence(model, array, sample_rate, 3)
        scores.append(greedy | {
            "agreement_wer": utterance_wer(greedy["text"], beam["text"]),
            "reference_wer": utterance_wer(row["sentence"], greedy["text"]),
        })
    good = pd.DataFrame(scores)
    good = good[good["reference_wer"] <= 0.25]
    if len(good) < 10:
        return {"average_logprob": -0.8, "no_speech_probability": 0.20, "agreement_wer": 0.20}
    return {
        "average_logprob": float(good["average_logprob"].quantile(0.10)),
        "no_speech_probability": float(min(0.30, good["no_speech_probability"].quantile(0.95))),
        "agreement_wer": float(min(0.25, good["agreement_wer"].quantile(0.95))),
    }


def pseudo_label_unlabeled(model_id: str) -> pd.DataFrame:
    from transformers import AutoModelForSpeechSeq2Seq
    model = AutoModelForSpeechSeq2Seq.from_pretrained(
        model_id, torch_dtype=torch.float16, low_cpu_mem_usage=True,
        token=cfg.hf_token, cache_dir=str(CACHE_DIR),
    ).to("cuda").eval()
    model.generation_config.forced_decoder_ids = None
    model.generation_config.task = "transcribe"
    thresholds = calibrate_pseudo_thresholds(model, load_waxal("validation"), cfg.pseudo_calibration_rows)
    print("Calibrated pseudo-label thresholds:", thresholds)
    stream = load_dataset("google/WaxalNLP", "dag_asr", split="unlabeled", streaming=True, token=cfg.hf_token)
    accepted, accepted_seconds = [], 0.0
    for index, row in enumerate(stream, 1):
        audio = row["audio"]
        if hasattr(audio, "get_all_samples"):
            samples = audio.get_all_samples()
            array = samples.data.detach().cpu().numpy().squeeze()
            sample_rate = int(samples.sample_rate)
        else:
            array, sample_rate = np.asarray(audio["array"]), int(audio["sampling_rate"])
        duration = len(array) / max(1, sample_rate)
        greedy = decode_with_confidence(model, array, sample_rate, 1)
        if greedy["average_logprob"] < thresholds["average_logprob"] or greedy["no_speech_probability"] > thresholds["no_speech_probability"]:
            continue
        beam = decode_with_confidence(model, array, sample_rate, 3)
        agreement = utterance_wer(greedy["text"], beam["text"])
        wps = len(normalize_eval(greedy["text"]).split()) / max(duration, 1e-6)
        if not (
            agreement <= thresholds["agreement_wer"]
            and 0.5 <= wps <= 5.0
            and compression_ratio(greedy["text"]) < 2.4
            and repetition_ratio(greedy["text"]) < 0.35
        ):
            continue
        sample_id = str(row.get("id", f"unlabeled-{index}"))
        confidence = float(np.exp(greedy["average_logprob"]) * (1.0 - greedy["no_speech_probability"]) * (1.0 - min(1.0, agreement)))
        accepted.append({
            "sample_id": sample_id, "source": "waxal_dag_asr", "source_version": "dag_asr",
            "audio_locator": f"hf://google/WaxalNLP/dag_asr/unlabeled/{sample_id}",
            "transcript_raw": greedy["text"], "transcript_eval": normalize_eval(greedy["text"]),
            "speaker_id": (
                f"waxal_dag_asr:{row.get('speaker_id')}"
                if str(row.get("speaker_id", "unknown")) != "unknown" else "unknown"
            ),
            "speaker_id_raw": str(row.get("speaker_id", "unknown")), "duration_s": duration,
            "domain": "spontaneous_pseudo", "split": "unlabeled", "license": "cc-by-4.0",
            "audio_hash": "", "is_pseudo": True, "pseudo_confidence": confidence,
            "average_logprob": greedy["average_logprob"], "no_speech_probability": greedy["no_speech_probability"],
            "greedy_beam_wer": agreement,
        })
        accepted_seconds += duration
        if len(accepted) % 50 == 0:
            print(f"Accepted {len(accepted):,} pseudo labels / {accepted_seconds / 3600:.2f}h after scanning {index:,} rows")
        if accepted_seconds >= cfg.pseudo_initial_hours * 3600:
            break
    del model
    torch.cuda.empty_cache()
    return pd.DataFrame(accepted)


if cfg.run_pseudo_labelling:
    if cfg.stage != "pseudo_label":
        raise RuntimeError("Set stage='pseudo_label' before generating pseudo labels")
    pseudo_df = pseudo_label_unlabeled(cfg.model_repo_id)
    if pseudo_df.empty:
        raise RuntimeError("No unlabeled rows passed calibrated confidence filters")
    pseudo_file = EVAL_DIR / "pseudo_manifest.parquet"
    pseudo_df.to_parquet(pseudo_file, index=False)
    from huggingface_hub import HfApi
    HfApi(token=cfg.hf_token).upload_file(
        repo_id=cfg.manifest_repo_id, repo_type="dataset", path_or_fileobj=str(pseudo_file),
        path_in_repo="phase1/pseudo_manifest.parquet", commit_message="Add confidence-filtered Dagbani pseudo labels",
    )
    print(f"Published {len(pseudo_df):,} pseudo labels covering {pseudo_df.duration_s.sum() / 3600:.2f}h")
else:
    print("Pseudo-labelling disabled. It is allowed only after the supervised model passes validation.")

## 12. Release, regression, and Medium-scaling gates

In [ ]:
def bootstrap_wer_difference(
    references: list[str], candidate: list[str], baseline: list[str],
    samples: int = 1_000, seed: int = 42,
) -> dict[str, float]:
    if not (len(references) == len(candidate) == len(baseline)):
        raise ValueError("Bootstrap inputs must have equal length")
    rng = np.random.default_rng(seed)
    differences = []
    for _ in range(samples):
        indices = rng.integers(0, len(references), size=len(references))
        refs = [normalize_eval(references[i]) for i in indices]
        cand = [normalize_eval(candidate[i]) for i in indices]
        base = [normalize_eval(baseline[i]) for i in indices]
        differences.append(float(jiwer.wer(refs, cand) - jiwer.wer(refs, base)))
    low, high = np.quantile(differences, [0.025, 0.975])
    return {"mean_difference": float(np.mean(differences)), "ci95_low": float(low), "ci95_high": float(high)}


def release_decision(candidate_metrics: dict, domain_regressions: dict[str, float], bootstrap: dict | None = None) -> dict:
    candidate_wer = float(candidate_metrics["normalized_wer"])
    beats_by_one_point = candidate_wer <= 0.33
    significant = bool(bootstrap and bootstrap["ci95_high"] < 0.0)
    domain_safe = all(regression <= 0.02 for regression in domain_regressions.values())
    return {
        "accepted": bool((beats_by_one_point or significant) and domain_safe),
        "candidate_wer": candidate_wer,
        "beats_34pct_by_one_point": beats_by_one_point,
        "bootstrap_supports_improvement": significant,
        "domain_regressions": domain_regressions,
        "research_only": True,
        "reason": "Native-speaker listening review is still required before production use.",
    }


def medium_gate(
    small_waxal_wer: float, small_pilot_wer: float, medium_pilot_wer: float,
    remaining_quota_hours: float, medium_projected_hours: float, medium_oom: bool,
) -> dict:
    checks = {
        "small_reproducible": small_waxal_wer <= 0.35,
        "matched_pilot_gain": medium_pilot_wer <= small_pilot_wer - 0.01,
        "fits_memory": not medium_oom,
        "fits_remaining_quota": medium_projected_hours + 1.0 <= remaining_quota_hours,
    }
    return {"allow_medium_full_run": all(checks.values()), "checks": checks}


def save_medium_gate(decision: dict, evidence: dict) -> Path:
    if not decision.get("allow_medium_full_run"):
        raise ValueError(f"Medium gate did not pass: {decision}")
    path = Path(cfg.medium_gate_file)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(decision | {"evidence": evidence}, indent=2), encoding="utf-8")
    print(f"Saved passing Medium authorization to {path}")
    return path


print("Acceptance gates loaded. Test is evaluated once, only after validation selects the final checkpoint.")
print("Reproduction target: WER <= 0.35 and CER <= 0.13. Release target: WER <= 0.33 or significant gain.")

## 13. Generate the research model card and run manifest

In [ ]:
def write_run_evidence(metrics: list[dict] | None = None) -> None:
    run_manifest = {
        "created_utc": pd.Timestamp.utcnow().isoformat(),
        "config": asdict(cfg) | {"hf_token": "<redacted>"},
        "phase1_readiness": readiness,
        "manifest_sha256": hashlib.sha256(pd.util.hash_pandas_object(manifest_df, index=True).values.tobytes()).hexdigest() if not manifest_df.empty else None,
        "metrics": metrics or [],
        "research_only": True,
    }
    (OUTPUT_DIR / "run_manifest.json").write_text(json.dumps(run_manifest, indent=2), encoding="utf-8")
    card = textwrap.dedent(f'''\
    ---
    language: dag
    license: other
    license_name: research-only pending corpus license review
    library_name: transformers
    pipeline_tag: automatic-speech-recognition
    ---

    # Dagbani Whisper ASR — Research Model

    Fine-tuned from `{cfg.base_model_id}` using the audited Dagbani manifest revision
    `{run_manifest['manifest_sha256']}`. The model is intended for research on
    conversational Dagbani ASR.

    ## Important limitation

    No native-speaker transcript/listening review was available for this run. The
    model must not be represented as production-ready or used for high-stakes
    decisions until native-speaker validation is completed.

    ## Transcript policy

    Ground truth was preserved. Evaluation normalization used only Unicode NFC,
    lowercase, whitespace cleanup, and punctuation removal; it did not infer tone
    or replace ambiguous Dagbani letters.
    ''').strip() + "\n"
    (OUTPUT_DIR / "README.md").write_text(card, encoding="utf-8")
    print(f"Run evidence written to {OUTPUT_DIR}")


write_run_evidence(generation_metrics + external_metrics)